# Assignment 4 

## Q1: 
Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries and other 2 entries constructed from your own roll number digits.

In [2]:
import pandas as pd

roll_number = "1024170434" 
d1 = int(roll_number[-2])
d2 = int(roll_number[-1])
categories = ["billing", "account", "general"]
cat1 = categories[d1 % 3] 
cat2 = categories[d2 % 3] 
print(f"Roll Number: {roll_number}")
print(f"Digit {d1} -> category {cat1}")
print(f"Digit {d2} -> category {cat2}")
fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.", "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.", "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.", "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.", "keywords": "pay payment upi fee", "category": "billing"},
]
extra_entry_1 = {"question": "how do I update my registered mobile number", "answer": "Update it in your profile settings.", "keywords": "update mobile phone number", "category": cat1}
extra_entry_2 = {"question": "where is the library located", "answer": "It is on the first floor of the main building.", "keywords": "library location where place", "category": cat2}
df = pd.DataFrame(fixed_entries + [extra_entry_1, extra_entry_2])
display(df)

Roll Number: 1024170434
Digit 3 -> category billing
Digit 4 -> category account


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do I update my registered mobile number,Update it in your profile settings.,update mobile phone number,billing
5,where is the library located,It is on the first floor of the main building.,library location where place,account


## Q2: 
Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [3]:
def score_hypothesis(query, df):
    query_words = set(query.lower().split())
    scores = []
    for idx, row in df.iterrows():
        entry_keywords = set(row['keywords'].lower().split())
        score = len(query_words.intersection(entry_keywords))
        scores.append(score)
    df_scored = df.copy()
    df_scored['score'] = scores
    df_ranked = df_scored.sort_values(by='score', ascending=False)
    return df_ranked[df_ranked['score'] > 0]
print("Query: 'how to reset login password'")
display(score_hypothesis("how to reset login password", df))

Query: 'how to reset login password'


,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,3


## Q3:
Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [4]:
def same_category(category_name, df):
    return df[df['category'] == category_name]['question'].tolist()
cat_to_test = cat1
print(f"Questions in category '{cat_to_test}':")
print(same_category(cat_to_test, df))

Questions in category 'billing':
['what is the annual fee', 'how can i pay the fee', 'how do I update my registered mobile number']


## Q4: 
Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [5]:
entry_index = 0
new_keyword = input("Enter a new keyword for the first entry: ")
df.at[entry_index, 'keywords'] = df.at[entry_index, 'keywords'] + " " + new_keyword
filename = f"{roll_number}_faq_data.csv"
df.to_csv(filename, index=False)
print(f"Saved updated dataframe to {filename}")
display(df.iloc[[entry_index]])

Saved updated dataframe to 1024170434_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge 1024,billing


## Q5: 
Using groupby, print how many FAQ entries you have per category.

In [6]:
category_counts = df.groupby('category').size()
print("FAQ entries per category:")
print(category_counts)

FAQ entries per category:
category
account    2
billing    3
general    1
dtype: int64


## Q6:
Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [ ]:
def score_hypothesis_with_ties(query, df):
    query_words = set(query.lower().split())
    
    scores = []
    for idx, row in df.iterrows():
        entry_keywords = set(row['keywords'].lower().split())
        score = len(query_words.intersection(entry_keywords))
        scores.append(score)    
    df_scored = df.copy()
    df_scored['score'] = scores
    df_matches = df_scored[df_scored['score'] > 0]
    if df_matches.empty:
        print("No matches found.")
        return df_matches
   
    max_score = df_matches['score'].max()
    best_matches = df_matches[df_matches['score'] == max_score]
    
    print(f"Found {len(best_matches)} top match(es) with score {max_score}:")
    for idx, row in best_matches.iterrows():
        print(f"- Q: {row['question']} (Score: {row['score']})")
        
    return best_matches
    
print("\n--- Testing query with a tie ('fee') ---")
display(score_hypothesis_with_ties("fee", df))
print("\n\n--- Testing query without a tie ('reset password') ---")
display(score_hypothesis_with_ties("reset password", df))


--- Testing query with a tie ('fee') ---
Found 2 top match(es) with score 1:
- Q: what is the annual fee (Score: 1)
- Q: how can i pay the fee (Score: 1)


,question,answer,keywords,category,score
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge 1024,billing,1
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing,1




--- Testing query without a tie ('reset password') ---
Found 1 top match(es) with score 2:
- Q: how to reset password (Score: 2)


,question,answer,keywords,category,score
1,how to reset password,Go to Settings > Reset Password.,password reset login,account,2
